# Day 4: Custom UI for the Voice Translator

Run the cells in order. Before you start: **Runtime → Change runtime type → T4 GPU**.

This notebook runs a **FastAPI backend** (your Python AI pipeline) and serves your **custom web UI** through a free public HTTPS link, so the microphone works on your laptop and phone.

## 1. Install everything
If a **Restart session** popup appears, click **Restart session**, then continue from step 2 (don't rerun this cell).

In [1]:
import os
os.environ["PYTHONHASHSEED"] = "0"

!pip install -q f5-tts faster-whisper fastapi uvicorn python-multipart
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
print("Install finished")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.1/143.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6

## 2. Load the AI models
Takes a few minutes the first time.

In [2]:
from faster_whisper import WhisperModel
from f5_tts.api import F5TTS

whisper_model = WhisperModel("medium", device="cuda", compute_type="float16")
f5tts = F5TTS()
print("Models loaded")

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Download Vocos from huggingface charactr/vocos-mel-24khz


config.yaml:   0%|          | 0.00/461 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 54.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

F5TTS_v1_Base/model_1250000.safetensors: reconstructing file:   0%|          |  0.00B / 1.35GB            

F5TTS_v1_Base/model_1250000.safetensors: downloading bytes:           |  0.00B            


vocab :  /usr/local/lib/python3.13/dist-packages/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors 

Models loaded


## 3. Upload your voice sample (consent clip)
Edit `sample_text` if your words were different.

In [3]:
from google.colab import files

uploaded = files.upload()
voice_sample = list(uploaded.keys())[0]
sample_text = "I agree to let Voice Translator use my voice to speak my translations. This is my real voice."
print("Voice sample:", voice_sample)

Saving sample.ogg to sample.ogg
Voice sample: sample.ogg


## 4. Save the web UI
This writes your custom interface to a file called `index.html`.

In [4]:
%%writefile index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>Voice Translator</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Public+Sans:wght@400;500;600;700&family=Noto+Sans+Devanagari:wght@400;600&family=Noto+Sans+Tamil:wght@400;600&family=Noto+Sans+Telugu:wght@400;600&display=swap" rel="stylesheet">
<script src="https://cdn.tailwindcss.com/3.4.16"></script>
<script>
  tailwind.config = {
    theme: {
      extend: {
        colors: {
          canvas: 'var(--color-bg)',
          surface: 'var(--color-surface)',
          ink: 'var(--color-text)',
          muted: 'var(--color-text-muted)',
          stroke: 'var(--color-border)',
          accent: 'var(--color-accent)',
          'on-accent': 'var(--color-on-accent)'
        },
        fontFamily: {
          ui: ['"Public Sans"', '"Noto Sans Devanagari"', '"Noto Sans Tamil"', '"Noto Sans Telugu"', 'system-ui', '-apple-system', '"Segoe UI"', 'sans-serif']
        }
      }
    }
  }
</script>
<style>
  /* ---------- Semantic color tokens ---------- */
  :root {
    --color-bg: #FFFFFF;
    --color-surface: #F5F5F7;
    --color-text: #111111;
    --color-text-muted: #5C5C63;
    --color-border: #D1D1D6;
    --color-accent: #0066CC;
    --color-on-accent: #FFFFFF;
    box-sizing: border-box;
    padding-top: env(safe-area-inset-top, 0px);
    padding-bottom: env(safe-area-inset-bottom, 0px);
  }
  @media (prefers-color-scheme: dark) {
    :root:not([data-theme="light"]) {
      --color-bg: #000000;
      --color-surface: #1C1C1E;
      --color-text: #FFFFFF;
      --color-text-muted: #A1A1A6;
      --color-border: #3A3A3C;
    }
  }
  :root[data-theme="dark"] {
    --color-bg: #000000;
    --color-surface: #1C1C1E;
    --color-text: #FFFFFF;
    --color-text-muted: #A1A1A6;
    --color-border: #3A3A3C;
  }
  html { scroll-padding-top: env(safe-area-inset-top, 0px); }
  body { margin: 0; background: var(--color-bg); color: var(--color-text); }

  :focus { outline: none; }
  :focus-visible { outline: 2px solid var(--color-accent); outline-offset: 3px; }

  /* ---------- State styling: only borders and fills change ---------- */
  [data-state="listening"] .source-card { border-color: var(--color-accent); }
  .status-live, .status-busy { display: none; }
  [data-state="listening"] .status-live { display: inline-flex; }
  [data-state="processing"] .status-busy { display: inline-flex; }
  [data-state="listening"] .status-text,
  [data-state="processing"] .status-text { display: none; }
  .icon-stop { display: none; }
  [data-state="listening"] .icon-mic { display: none; }
  [data-state="listening"] .icon-stop { display: block; }
  [data-state="processing"] .mic-button { cursor: wait; }
  .status-text[data-tone="error"] { color: var(--color-text); font-weight: 600; }
  [data-empty="true"] { color: var(--color-text-muted); font-weight: 400; }

  .level-bar { transform-origin: bottom; animation: level 0.9s ease-in-out infinite; }
  .level-bar:nth-child(2) { animation-delay: 0.15s; }
  .level-bar:nth-child(3) { animation-delay: 0.3s; }
  @keyframes level { 0%, 100% { transform: scaleY(0.35); } 50% { transform: scaleY(1); } }
  @media (prefers-reduced-motion: reduce) { .level-bar { animation: none; } }
</style>
</head>
<body class="font-ui antialiased">

<main class="app mx-auto flex min-h-screen max-w-md flex-col gap-3 px-4 py-5" data-state="idle">

  <header class="flex items-center justify-between px-1 pb-1">
    <h1 class="text-[20px] font-bold">Voice Translator</h1>
    <button type="button" id="theme-toggle" class="flex h-11 w-11 items-center justify-center rounded-full border border-stroke bg-surface text-ink" aria-label="Switch theme">
      <svg class="h-5 w-5" viewBox="0 0 20 20" fill="none" stroke="currentColor" stroke-width="1.75" aria-hidden="true"><circle cx="10" cy="10" r="7"/><path d="M10 3a7 7 0 0 0 0 14z" fill="currentColor"/></svg>
    </button>
  </header>

  <!-- Source: speaker -->
  <section class="source-card flex min-h-[176px] flex-col rounded-2xl border border-stroke bg-surface p-4" aria-labelledby="speaker-label">
    <div class="flex items-center justify-between gap-3">
      <span id="speaker-label" class="text-[13px] font-medium text-muted">Speaker</span>
      <div class="relative">
        <label class="sr-only" for="source-lang">Speaker language</label>
        <select id="source-lang" class="h-11 appearance-none rounded-full border border-stroke bg-canvas pl-4 pr-10 text-[15px] font-semibold text-ink">
          <option value="hi" selected>Hindi</option>
          <option value="ta">Tamil</option>
          <option value="te">Telugu</option>
        </select>
        <svg class="pointer-events-none absolute right-4 top-1/2 h-4 w-4 -translate-y-1/2" viewBox="0 0 16 16" fill="none" stroke="currentColor" stroke-width="1.75" aria-hidden="true"><path d="M4 6l4 4 4-4" stroke-linecap="round" stroke-linejoin="round"/></svg>
      </div>
    </div>

    <p id="source-text" class="mt-4 flex-1 text-[20px] leading-snug" data-empty="true">Your words will appear here.</p>

    <div class="mt-3 flex items-center gap-2 text-[13px]">
      <span id="status-text" class="status-text text-muted">Tap the microphone and speak</span>
      <span class="status-live items-center gap-2 rounded-full bg-accent px-3 py-1 font-semibold text-on-accent">
        <span class="flex h-3 items-end gap-[2px]" aria-hidden="true">
          <span class="level-bar block h-3 w-[3px] rounded-sm bg-on-accent"></span>
          <span class="level-bar block h-3 w-[3px] rounded-sm bg-on-accent"></span>
          <span class="level-bar block h-3 w-[3px] rounded-sm bg-on-accent"></span>
        </span>
        Listening, tap to stop
      </span>
      <span class="status-busy items-center rounded-full bg-accent px-3 py-1 font-semibold text-on-accent">Translating</span>
    </div>
  </section>

  <!-- Control strip -->
  <div class="grid grid-cols-[1fr_auto_1fr] items-center py-2">
    <div class="flex justify-start">
      <button type="button" id="clear-btn" aria-label="Clear" class="flex h-12 w-12 items-center justify-center rounded-full border border-stroke bg-surface text-ink">
        <svg class="h-5 w-5" viewBox="0 0 20 20" fill="none" stroke="currentColor" stroke-width="1.75" aria-hidden="true"><path d="M5 5l10 10M15 5L5 15" stroke-linecap="round"/></svg>
      </button>
    </div>

    <button type="button" id="mic-btn" aria-pressed="false" aria-label="Start listening" class="mic-button flex h-24 w-24 items-center justify-center rounded-full bg-accent text-on-accent">
      <svg class="icon-mic h-10 w-10" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" aria-hidden="true"><rect x="9" y="3" width="6" height="11" rx="3"/><path d="M5 11a7 7 0 0 0 14 0M12 18v3" stroke-linecap="round"/></svg>
      <svg class="icon-stop h-8 w-8" viewBox="0 0 24 24" fill="currentColor" aria-hidden="true"><rect x="5" y="5" width="14" height="14" rx="2.5"/></svg>
    </button>

    <div class="flex flex-col items-end text-right">
      <span id="latency" class="text-[17px] font-semibold tabular-nums">None yet</span>
      <span class="text-[12px] text-muted">last translation</span>
    </div>
  </div>

  <!-- Target: listener -->
  <section class="flex min-h-[176px] flex-col rounded-2xl border border-stroke bg-surface p-4" aria-labelledby="listener-label">
    <div class="flex items-center justify-between gap-3">
      <span id="listener-label" class="text-[13px] font-medium text-muted">Listener</span>
      <div class="relative">
        <label class="sr-only" for="target-lang">Listener language</label>
        <select id="target-lang" class="h-11 appearance-none rounded-full border border-stroke bg-canvas pl-4 pr-10 text-[15px] font-semibold text-ink">
          <option value="en" selected>English</option>
        </select>
        <svg class="pointer-events-none absolute right-4 top-1/2 h-4 w-4 -translate-y-1/2" viewBox="0 0 16 16" fill="none" stroke="currentColor" stroke-width="1.75" aria-hidden="true"><path d="M4 6l4 4 4-4" stroke-linecap="round" stroke-linejoin="round"/></svg>
      </div>
    </div>

    <p id="target-text" class="mt-4 flex-1 text-[22px] font-semibold leading-snug" lang="en" data-empty="true">The English translation will appear here.</p>

    <div class="mt-3 flex gap-2">
      <button type="button" id="play-btn" disabled class="inline-flex h-11 items-center gap-2 rounded-full border border-stroke bg-canvas px-4 text-[15px] font-semibold text-ink disabled:cursor-not-allowed disabled:text-muted">
        <svg class="h-4 w-4" viewBox="0 0 16 16" fill="currentColor" aria-hidden="true"><path d="M4 2.8v10.4a.6.6 0 0 0 .9.5l8.3-5.2a.6.6 0 0 0 0-1L4.9 2.3a.6.6 0 0 0-.9.5z"/></svg>
        <span id="play-label">Play in my voice</span>
      </button>
      <button type="button" id="copy-btn" disabled class="inline-flex h-11 items-center rounded-full border border-stroke bg-canvas px-4 text-[15px] font-semibold text-ink disabled:cursor-not-allowed disabled:text-muted">Copy text</button>
    </div>
  </section>

  <p class="sr-only" aria-live="polite" id="announce"></p>
</main>

<script>
  // ---------- Element references ----------
  var app = document.querySelector('.app');
  var micBtn = document.getElementById('mic-btn');
  var clearBtn = document.getElementById('clear-btn');
  var playBtn = document.getElementById('play-btn');
  var playLabel = document.getElementById('play-label');
  var copyBtn = document.getElementById('copy-btn');
  var sourceSelect = document.getElementById('source-lang');
  var targetSelect = document.getElementById('target-lang');
  var sourceText = document.getElementById('source-text');
  var targetText = document.getElementById('target-text');
  var statusText = document.getElementById('status-text');
  var latency = document.getElementById('latency');
  var announce = document.getElementById('announce');
  var themeToggle = document.getElementById('theme-toggle');

  var DEFAULT_STATUS = 'Tap the microphone and speak';
  var recorder = null;
  var stream = null;
  var chunks = [];
  var translatedAudio = null;

  // ---------- UI state ----------
  function setState(state, message, tone) {
    app.dataset.state = state;
    micBtn.setAttribute('aria-pressed', String(state === 'listening'));
    micBtn.setAttribute('aria-label', state === 'listening' ? 'Stop and translate' : 'Start listening');
    micBtn.disabled = state === 'processing';
    statusText.textContent = message || DEFAULT_STATUS;
    statusText.dataset.tone = tone || '';
    if (state === 'listening') announce.textContent = 'Listening';
    else if (state === 'processing') announce.textContent = 'Translating';
    else announce.textContent = message || '';
  }

  function setText(el, text, placeholder) {
    if (text) { el.textContent = text; el.dataset.empty = 'false'; }
    else { el.textContent = placeholder; el.dataset.empty = 'true'; }
  }

  // ---------- Recording ----------
  async function startRecording() {
    try {
      stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    } catch (err) {
      var msg = err && err.name === 'NotAllowedError'
        ? 'Microphone blocked. Allow microphone access in your browser settings, then try again.'
        : 'No microphone found. Connect one, or open this page on your phone.';
      setState('idle', msg, 'error');
      return;
    }
    chunks = [];
    recorder = new MediaRecorder(stream);
    recorder.ondataavailable = function (e) { if (e.data && e.data.size) chunks.push(e.data); };
    recorder.onstop = sendRecording;
    recorder.start();
    setState('listening');
  }

  function stopRecording() {
    setState('processing');
    recorder.stop();
    stream.getTracks().forEach(function (t) { t.stop(); });
  }

  // ---------- Talk to the Python backend ----------
  async function sendRecording() {
    var type = recorder.mimeType || 'audio/webm';
    var ext = type.indexOf('mp4') !== -1 ? 'mp4' : (type.indexOf('ogg') !== -1 ? 'ogg' : 'webm');
    var form = new FormData();
    form.append('audio', new Blob(chunks, { type: type }), 'speech.' + ext);
    form.append('source', sourceSelect.value);
    form.append('target', targetSelect.value);

    var started = performance.now();
    try {
      var res = await fetch('/api/translate', { method: 'POST', body: form });
      var data = await res.json();
      if (!res.ok) throw new Error(data.error || 'Translation failed. Try again.');

      var seconds = (performance.now() - started) / 1000;
      latency.textContent = seconds.toFixed(1) + ' s';
      sourceText.lang = sourceSelect.value;
      setText(sourceText, data.source_text, 'Your words will appear here.');
      setText(targetText, data.english_text, 'The English translation will appear here.');

      translatedAudio = new Audio(data.audio_url);
      translatedAudio.addEventListener('ended', function () { playLabel.textContent = 'Play in my voice'; });
      playBtn.disabled = false;
      copyBtn.disabled = false;
      setState('idle', 'Done. Tap the microphone to translate again.');
      playTranslation();
    } catch (err) {
      var message = err instanceof TypeError
        ? 'Cannot reach the translator. Check that the Colab notebook is still running.'
        : err.message;
      setState('idle', message, 'error');
    }
  }

  function playTranslation() {
    if (!translatedAudio) return;
    translatedAudio.currentTime = 0;
    translatedAudio.play().then(function () {
      playLabel.textContent = 'Playing';
    }).catch(function () {
      playLabel.textContent = 'Play in my voice';
    });
  }

  // ---------- Buttons ----------
  micBtn.addEventListener('click', function () {
    if (app.dataset.state === 'listening') stopRecording();
    else if (app.dataset.state === 'idle') startRecording();
  });

  playBtn.addEventListener('click', playTranslation);

  copyBtn.addEventListener('click', function () {
    try {
      navigator.clipboard.writeText(targetText.textContent).then(function () {
        copyBtn.textContent = 'Copied';
      }, function () { copyBtn.textContent = 'Copy blocked'; });
    } catch (e) { copyBtn.textContent = 'Copy blocked'; }
    setTimeout(function () { copyBtn.textContent = 'Copy text'; }, 1500);
  });

  clearBtn.addEventListener('click', function () {
    if (app.dataset.state !== 'idle') return;
    setText(sourceText, '', 'Your words will appear here.');
    setText(targetText, '', 'The English translation will appear here.');
    translatedAudio = null;
    playBtn.disabled = true;
    copyBtn.disabled = true;
    setState('idle', 'Cleared.');
  });

  // ---------- Theme toggle ----------
  function currentTheme() {
    var set = document.documentElement.dataset.theme;
    if (set) return set;
    return window.matchMedia('(prefers-color-scheme: dark)').matches ? 'dark' : 'light';
  }
  function applyTheme(theme) {
    document.documentElement.dataset.theme = theme;
    themeToggle.setAttribute('aria-label', theme === 'dark' ? 'Switch to light theme' : 'Switch to dark theme');
  }
  try {
    var saved = localStorage.getItem('vt-theme');
    if (saved) applyTheme(saved);
    else themeToggle.setAttribute('aria-label', currentTheme() === 'dark' ? 'Switch to light theme' : 'Switch to dark theme');
  } catch (e) {}
  themeToggle.addEventListener('click', function () {
    var next = currentTheme() === 'dark' ? 'light' : 'dark';
    applyTheme(next);
    try { localStorage.setItem('vt-theme', next); } catch (e) {}
  });
</script>
</body>
</html>


Writing index.html


## 5. The backend
Defines the API the web page talks to. No output is expected.

In [7]:
import os
import time
import uuid
import tempfile
import threading
from pathlib import Path

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse, JSONResponse

AUDIO_DIR = Path("generated_audio")
AUDIO_DIR.mkdir(exist_ok=True)
SUPPORTED_SOURCES = {"hi", "ta", "te"}
gpu_lock = threading.Lock()  # one request uses the GPU at a time

api = FastAPI()


@api.get("/")
def home():
    return FileResponse("index.html")


@api.post("/api/translate")
def translate(
    audio: UploadFile = File(...),
    source: str = Form("hi"),
    target: str = Form("en"),
):
    if source not in SUPPORTED_SOURCES:
        return JSONResponse({"error": "That speaker language is not supported yet."}, status_code=400)
    if target != "en":
        return JSONResponse({"error": "Only English output is available right now."}, status_code=400)

    # Save the uploaded recording to a temporary file
    suffix = Path(audio.filename or "speech.webm").suffix or ".webm"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(audio.file.read())
        input_path = tmp.name

    with gpu_lock:
        # 1. Speech recognition: original words, then English translation
        start = time.time()
        segments, _ = whisper_model.transcribe(
            input_path, language=source, task="transcribe", vad_filter=True, beam_size=1
        )
        source_text = " ".join(s.text.strip() for s in segments)
        segments, _ = whisper_model.transcribe(
            input_path, language=source, task="translate", vad_filter=True, beam_size=1
        )
        english_text = " ".join(s.text.strip() for s in segments)
        asr_time = time.time() - start
        os.remove(input_path)

        if not english_text:
            return JSONResponse(
                {"error": "No speech detected. Speak a little closer to the microphone and try again."},
                status_code=422,
            )

        # 2. Voice cloning: speak the English text in your voice
        start = time.time()
        audio_id = uuid.uuid4().hex
        output_path = AUDIO_DIR / f"{audio_id}.wav"
        f5tts.infer(
            ref_file=voice_sample,
            ref_text=sample_text,
            gen_text=english_text,
            file_wave=str(output_path),
            nfe_step=32,
        )
        tts_time = time.time() - start

    print(f"ASR {asr_time:.1f}s | TTS {tts_time:.1f}s | {english_text}")
    return {
        "source_text": source_text,
        "english_text": english_text,
        "audio_url": f"/api/audio/{audio_id}",
        "timings": {"asr": round(asr_time, 2), "tts": round(tts_time, 2)},
    }


@api.get("/api/audio/{audio_id}")
def get_audio(audio_id: str):
    path = AUDIO_DIR / f"{audio_id}.wav"
    if not audio_id.isalnum() or not path.exists():
        return JSONResponse({"error": "Audio not found."}, status_code=404)
    return FileResponse(path, media_type="audio/wav")

## 6. Start the server and get your public link
Open the link it prints (on your laptop or phone). If the page doesn't load right away, wait 10 seconds and refresh.

Changed the backend or UI? Rerun that cell, then rerun this one to restart the server.

In [8]:
import re
import time
import subprocess
import threading
import uvicorn

# Stop any server and tunnel from a previous run of this cell
if "server" in globals():
    server.should_exit = True
    time.sleep(2)
if "tunnel" in globals():
    tunnel.terminate()

config = uvicorn.Config(api, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
for line in tunnel.stdout:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

# Keep reading the tunnel's logs in the background so it never stalls
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

print("Open your app here:", public_url)

Open your app here: https://offshore-reference-particles-temperatures.trycloudflare.com
